# MMF Series Profiling
Computes statistical properties for each time series and classifies them.

In [ ]:
%pip install statsmodels scipy --quiet

In [ ]:
%restart_python

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
use_case = "{use_case}"
train_table = "{train_table}"
freq = "{freq}"
prediction_length = {prediction_length}

In [ ]:
df = spark.table(f"{catalog}.{schema}.{train_table}")
print(f"Loaded {df.count()} rows for use case: {use_case}")

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType


def profile_series(pdf: pd.DataFrame) -> pd.DataFrame:
    """Compute statistical properties for a single time series."""
    uid = pdf["unique_id"].iloc[0]
    y = pdf.sort_values("ds")["y"].dropna().values
    n = len(y)

    result = {
        "unique_id": uid,
        "series_length": n,
        "adf_pvalue": None,
        "seasonality_strength": 0.0,
        "trend_strength": 0.0,
        "spectral_entropy": 1.0,
        "autocorrelation_lag1": 0.0,
        "snr": 0.0,
        "sparsity": 0.0,
        "cv": 0.0,
    }

    if n < 6:
        return pd.DataFrame([result])

    # Stationarity — ADF test
    try:
        from statsmodels.tsa.stattools import adfuller
        adf_result = adfuller(y, autolag="AIC")
        result["adf_pvalue"] = float(adf_result[1])
    except Exception:
        result["adf_pvalue"] = 1.0

    # Seasonality & trend strength via STL
    try:
        from statsmodels.tsa.seasonal import STL
        freq_map = {"H": 24, "D": 7, "W": 52, "M": 12}
        period = freq_map.get(freq, 7)
        if n >= 2 * period:
            stl = STL(y, period=period, robust=True).fit()
            var_resid = np.var(stl.resid)
            var_deseason = np.var(y - stl.seasonal)
            var_detrend = np.var(y - stl.trend)
            result["seasonality_strength"] = float(max(0, 1 - var_resid / max(var_deseason, 1e-10)))
            result["trend_strength"] = float(max(0, 1 - var_resid / max(var_detrend, 1e-10)))
    except Exception:
        pass

    # Spectral entropy
    try:
        from scipy.signal import periodogram
        from scipy.stats import entropy as sp_entropy
        _, psd = periodogram(y)
        psd_norm = psd / (psd.sum() + 1e-10)
        max_entropy = np.log(len(psd_norm)) if len(psd_norm) > 0 else 1.0
        result["spectral_entropy"] = float(sp_entropy(psd_norm + 1e-10) / max(max_entropy, 1e-10))
    except Exception:
        pass

    # Autocorrelation at lag 1
    try:
        result["autocorrelation_lag1"] = float(pd.Series(y).autocorr(lag=1))
    except Exception:
        pass

    # Signal-to-noise ratio
    mean_y = np.mean(y)
    var_y = np.var(y)
    result["snr"] = float(mean_y ** 2 / max(var_y, 1e-10))

    # Sparsity
    result["sparsity"] = float(np.mean(np.abs(y) < 1e-8))

    # Coefficient of variation
    std_y = np.std(y)
    result["cv"] = float(std_y / max(abs(mean_y), 1e-10))

    return pd.DataFrame([result])

In [ ]:
output_schema = StructType([
    StructField("unique_id", StringType()),
    StructField("series_length", IntegerType()),
    StructField("adf_pvalue", DoubleType()),
    StructField("seasonality_strength", DoubleType()),
    StructField("trend_strength", DoubleType()),
    StructField("spectral_entropy", DoubleType()),
    StructField("autocorrelation_lag1", DoubleType()),
    StructField("snr", DoubleType()),
    StructField("sparsity", DoubleType()),
    StructField("cv", DoubleType()),
])

profile_df = df.groupby("unique_id").applyInPandas(profile_series, schema=output_schema)
print(f"Profiled {profile_df.count()} series")
profile_df.show(10, truncate=False)

In [ ]:
from pyspark.sql import functions as F

classified_df = profile_df.withColumn(
    "forecastability_class",
    F.when(
        (F.col("spectral_entropy") < 0.8)
        & (F.col("series_length") >= 2 * prediction_length)
        & (F.col("sparsity") < 0.5)
        & (F.col("snr") > 0.1),
        F.lit("high_confidence"),
    ).otherwise(F.lit("low_signal")),
)


def recommend_models(row):
    cls = row["forecastability_class"]
    if cls == "low_signal":
        return "StatsForecastBaselineNaive,StatsForecastBaselineSeasonalNaive"

    sparsity = row["sparsity"] or 0.0
    seasonality = row["seasonality_strength"] or 0.0
    trend = row["trend_strength"] or 0.0
    entropy = row["spectral_entropy"] or 0.0
    length = row["series_length"] or 0

    if sparsity > 0.3:
        return "StatsForecastTSB,StatsForecastADIDA,StatsForecastIMAPA,StatsForecastCrostonClassic"
    if seasonality > 0.6 and (row["adf_pvalue"] or 1.0) < 0.05:
        return "StatsForecastAutoArima,StatsForecastAutoETS,StatsForecastAutoTheta,ChronosBoltBase,Chronos2,TimesFM_2_5_200m"
    if trend > 0.6 and seasonality <= 0.6:
        return "StatsForecastAutoArima,SKTimeProphet,NeuralForecastAutoNHITS,ChronosBoltBase,Chronos2,TimesFM_2_5_200m"
    if entropy > 0.6 and length > 200:
        return "NeuralForecastAutoNHITS,NeuralForecastAutoPatchTST,ChronosBoltBase,Chronos2,TimesFM_2_5_200m"
    if length < 50:
        return "StatsForecastAutoETS,StatsForecastAutoCES,ChronosBoltBase,Chronos2,TimesFM_2_5_200m"
    return "StatsForecastAutoArima,NeuralForecastAutoNHITS,ChronosBoltBase,Chronos2,TimesFM_2_5_200m"


from pyspark.sql.functions import udf
from pyspark.sql.types import StringType as ST

recommend_udf = udf(recommend_models, ST())
classified_df = classified_df.withColumn("recommended_models", recommend_udf(F.struct(*classified_df.columns)))


def derive_model_types(models_str):
    if models_str is None:
        return "local"
    types = set()
    for m in models_str.split(","):
        m = m.strip()
        if m.startswith("NeuralForecast"):
            types.add("global")
        elif m.startswith("Chronos") or m.startswith("TimesFM"):
            types.add("foundation")
        else:
            types.add("local")
    return ",".join(sorted(types))


derive_types_udf = udf(derive_model_types, ST())
classified_df = classified_df.withColumn("model_types_needed", derive_types_udf(F.col("recommended_models")))

In [ ]:
output_table = f"{catalog}.{schema}.{use_case}_series_profile"
classified_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)

total = classified_df.count()
high = classified_df.filter(F.col("forecastability_class") == "high_confidence").count()
low = classified_df.filter(F.col("forecastability_class") == "low_signal").count()
print(f"\nProfiling complete.")
print(f"  Total series: {total}")
print(f"  High-confidence: {high}")
print(f"  Low-signal: {low}")
print(f"  Output: {output_table}")